In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## **Read our [Gemma 3 blog](https://unsloth.ai/blog/gemma3) for what's new in Unsloth and our [Reasoning blog](https://unsloth.ai/blog/r1-reasoning) on how to train reasoning models.**

## Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


## 1. Installation

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    !pip install --no-deps unsloth vllm
# Install latest Hugging Face for Gemma-3!
!pip install --no-deps git+https://github.com/huggingface/transformers@v4.49.0-Gemma-3
!pip install bitsandbytes
!pip install unsloth_zoo
!pip install msgspec
!pip install blake3
!pip install gguf
!pip install networkx==2.8.8 --force-reinstall

In [ ]:
#@title Colab Extra Install { display-mode: "form" }
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth vllm
else:
    !pip install --no-deps unsloth vllm
    # [NOTE] Do the below ONLY in Colab! Use [[pip install unsloth vllm]]
    # Skip restarting message in Colab
    import sys, re, requests; modules = list(sys.modules.keys())
    for x in modules: sys.modules.pop(x) if "PIL" in x or "google" in x else None
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf datasets huggingface_hub hf_transfer

    # vLLM requirements - vLLM breaks Colab due to reinstalling numpy
    f = requests.get("https://raw.githubusercontent.com/vllm-project/vllm/refs/heads/main/requirements/common.txt").content
    with open("vllm_requirements.txt", "wb") as file:
        file.write(re.sub(rb"(transformers|numpy|xformers)[^\n]{1,}\n", b"", f))
    !pip install -r vllm_requirements.txt

In [ ]:
%%capture
!pip uninstall -y pillow
!pip install --no-cache-dir --force-reinstall pillow==11.3.0

### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

In [ ]:
from unsloth import FastModel
import torch

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-1b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",

    # Other popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastModel.from_pretrained(
    model_name = "unsloth/gemma-3-4b-it",
    max_seq_length = 2048, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    load_in_8bit = False, # [NEW!] A bit more accurate, uses 2x memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1112 15:14:34.611000 14533 torch/utils/_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 15:14:34.674000 14533 torch/utils/_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1112 15:14:38.494000 14533 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1112 15:14:38.496000 14533 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1112 15:14:38.498000 14533 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.11.4: Fast Gemma3 patching. Transformers: 5.17.0. vLLM: 0.33.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.14.1+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.8.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Using float16 precision for gemma3 won't work! Using float32.


Loading weights:   0%|          | 0/883 [00:00<?, ?it/s]

Skipping model.vision_tower.encoder.layers.0.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.0.mlp.fc2: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.k_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.v_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.q_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.self_attn.out_proj: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc1: no quant_state found
Skipping model.vision_tower.encoder.layers.1.mlp.fc2: no quant_state found
Skipping model.vision_to

We now add LoRA adapters so we only need to update a small amount of parameters!

In [ ]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False, # Turn off for just text!
    finetune_language_layers   = True,  # Should leave on!
    finetune_attention_modules = True,  # Attention good for GRPO
    finetune_mlp_modules       = True,  # SHould leave on always!

    r = 8,           # Larger = higher accuracy, but might overfit
    lora_alpha = 8,  # Recommended alpha == r at least
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

<a name="Data"></a>
### Data Prep
We now use the `Gemma-3` format for conversation style finetunes. We use [Maxime Labonne's FineTome-100k](https://huggingface.co/datasets/mlabonne/FineTome-100k) dataset in ShareGPT style. Gemma-3 renders multi turn conversations like below:

```
<bos><start_of_turn>user
Hello!<end_of_turn>
<start_of_turn>model
Hey there!<end_of_turn>
```

We use our `get_chat_template` function to get the correct chat template. We support `zephyr, chatml, mistral, llama, alpaca, vicuna, vicuna_old, phi3, llama3, phi4, qwen2.5, gemma3` and more.

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)

In [64]:
from datasets import load_dataset

dataset = load_dataset(
    'csv',
    data_files="dataasilida_cleaned.csv"
)


# Access the 'train' split and display the first few rows using pandas
print(dataset['train'].to_pandas().head())

                                        input,output  \
0  ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ...   
1  លោកអ្នកទទួលបានអត្ថប្រយោជន៍ពីការប្រើប្រាស់ ប័ណ្...   
2  លោកអ្នកអាចស្នើសុំ ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដ...   
3  លោកអ្នកអាចដកប្រាក់បានរហូតដល់ ១០០០០ដុល្លារអាមេរ...   
4  ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពលភា...   

                                   __index_level_0__  
0      តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី  
1  តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូ...  
2  តើធ្វើដូចម្តេចទើបខ្ញុំអាច ស្នើសុំប័ណ្ណឥណពន្ធចេ...  
3  តើខ្ញុំអាចប្រើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ដក...  
4  តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពល...  


In [65]:
# Rename columns safely
if "input,output" in dataset["train"].column_names:
    dataset = dataset.rename_column("input,output", "input")
if "__index_level_0__" in dataset["train"].column_names:
    dataset = dataset.rename_column("__index_level_0__", "output")
print(dataset['train'].to_pandas().head())

                                               input  \
0  ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ...   
1  លោកអ្នកទទួលបានអត្ថប្រយោជន៍ពីការប្រើប្រាស់ ប័ណ្...   
2  លោកអ្នកអាចស្នើសុំ ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដ...   
3  លោកអ្នកអាចដកប្រាក់បានរហូតដល់ ១០០០០ដុល្លារអាមេរ...   
4  ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពលភា...   

                                              output  
0      តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី  
1  តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូ...  
2  តើធ្វើដូចម្តេចទើបខ្ញុំអាច ស្នើសុំប័ណ្ណឥណពន្ធចេ...  
3  តើខ្ញុំអាចប្រើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ដក...  
4  តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពល...  


In [66]:
import pandas as pd

# Convert Hugging Face dataset split to a pandas DataFrame for analysis
df = dataset['train'].to_pandas()

print("1. Checking for True Missing Values (NaN/None)")
print(df.isnull().sum())
print("\n")

print("2. Checking for Empty Spaces (Blank/Whitespace Strings)")
# Check if cells are empty string or contain only whitespace
empty_spaces_input = df['input'].apply(lambda x: str(x).strip() == '').sum() if 'input' in df.columns else 0
empty_spaces_output = df['output'].apply(lambda x: str(x).strip() == '').sum() if 'output' in df.columns else 0

print(f"Empty spaces in 'input': {empty_spaces_input}")
print(f"Empty spaces in 'output': {empty_spaces_output}")

print("\n" )
print("3. Checking for Duplicate Rows")
duplicate_count = df.duplicated().sum()
print(f"Total fully duplicate rows: {duplicate_count}")


1. Checking for True Missing Values (NaN/None)
input     7
output    0
dtype: int64


2. Checking for Empty Spaces (Blank/Whitespace Strings)
Empty spaces in 'input': 0
Empty spaces in 'output': 0


3. Checking for Duplicate Rows
Total fully duplicate rows: 1


In [67]:
# 1. Convert back to pandas to easily drop duplicates and NaNs
df_cleaned = dataset['train'].to_pandas()

# 2. Drop any rows where 'input' or 'output' is missing (NaN)
df_cleaned = df_cleaned.dropna(subset=['input', 'output'])

# 3. Drop duplicate rows
df_cleaned = df_cleaned.drop_duplicates()

# 4. Convert the cleaned pandas DataFrame back into a Hugging Face Dataset split
from datasets import Dataset, DatasetDict
cleaned_dataset = Dataset.from_pandas(df_cleaned, preserve_index=False)

# Overwrite our dataset dictionary's train split with the cleaned dataset
dataset['train'] = cleaned_dataset

print("Dataset Cleaned Successfully")
print(f"New dataset row count: {len(dataset['train'])}")
print(dataset['train'].to_pandas().isnull().sum())

print("\nSample")
print(dataset['train'].to_pandas().head())

Dataset Cleaned Successfully
New dataset row count: 513
input     0
output    0
dtype: int64

Sample
                                               input  \
0  ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា គឺជាប័ណ្ណ...   
1  លោកអ្នកទទួលបានអត្ថប្រយោជន៍ពីការប្រើប្រាស់ ប័ណ្...   
2  លោកអ្នកអាចស្នើសុំ ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដ...   
3  លោកអ្នកអាចដកប្រាក់បានរហូតដល់ ១០០០០ដុល្លារអាមេរ...   
4  ប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពលភា...   

                                              output  
0      តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ជាអ្វី  
1  តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា ផ្តល់នូ...  
2  តើធ្វើដូចម្តេចទើបខ្ញុំអាច ស្នើសុំប័ណ្ណឥណពន្ធចេ...  
3  តើខ្ញុំអាចប្រើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ដក...  
4  តើប័ណ្ណឥណពន្ធចេស៊ីប៊ី អេស៊ីលីដា ធម្មតា មានសុពល...  


In [68]:
dataset["train"].to_csv("output.csv", index=False)

Creating CSV from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

568619

### Split the Dataset into Train and Validation Sets
Splitting the dataset ensures we can evaluate how well our model learns. We will use Hugging Face's native `.train_test_split` method to split 80% for training and 20% for validation.

In [56]:
# Perform an 80/20 train/test split on our cleaned Hugging Face dataset
splitted_dataset = dataset['train'].train_test_split(test_size=0.2, seed=3407)

# This gives us a DatasetDict with 'train' and 'test' keys
dataset = splitted_dataset
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['input', 'output'],
        num_rows: 414
    })
    test: Dataset({
        features: ['input', 'output'],
        num_rows: 104
    })
})


In [57]:
df = dataset['train'].to_pandas()
display(df.head(10))
dataset['train']['input'][0]

,input,output
0,តើលក្ខណសម្បត្តិអតិថិជនបែបណា ដើម្បីទទួលបានឥណទាន...,\nមានទីលំនៅអចិន្រ្តៃយ៍ ដោយមានឯកសារបញ្ជាក់អត្តស...
1,ដូចម្តេចដែលហៅថាសេវាទិញមូលប្បទានប័ត្របរទេស,សេវាទិញមូលប្បទានប័ត្របរទេស គឺជាសេវាមួយប្រភេទក្...
2,តើរបៀបសងត្រឡប់របស់ ឥណទានទុនបង្វិលយ៉ាងដូចម្តេចដែរ,ការសងសងត្រឡប់របស់ ឥណទានទុនបង្វិល អាចធ្វើឡើងដូច...
3,តើពេលបើកគណនីដំបូងត្រូវការតម្កល់ប្រាក់ឬទេ,សម្រាប់ការបើកគណនីបញ្ញើសំចៃ មិនតម្រូវឲ្យដាក់ប្រ...
4,តើអតិថិជនត្រូវមានលក្ខខណ្ឌដូចម្តេច ក្នុងការប្រើ...,អតិថិជនត្រូវមានលក្ខខណ្ឌដូចខាងក្រោម៖ \n ចំពោះឯក...
5,តើខ្ញុំនឹងទទួលបានការប្រាក់ ពីការដាក់ប្រាក់ចូលទ...,មិនមានការប្រាក់សម្រាប់ការដាក់ប្រាក់ទៅក្នុងគណនី...
6,តើវិធីសាស្ត្រទូទាត់នៅក្នុងលិខិតឥណទានមានប៉ុន្មា...,វិធីសាស្ត្រទូទាត់នៅក្នុងលិខិតឥណទានមាន ៣ ប្រភេទ...
7,តើរបៀបសងរបស់ឥណទាននិស្សិត យ៉ាងដូម្តេចដែរ,ការសងប្រាក់ត្រឡប់មកវិញមានពីររបៀប៖ \n សងការប្រា...
8,ដើម្បីបើកគណនីឯកត្តជនបានត្រូវមានឯកសារអ្វីខ្លះ,ត្រូវមានឯកសារដូចខាងក្រោម៖ \n អត្តសញ្ញាណប័ណ្ណសញ...
9,តើខ្ញុំអាចបញ្ឈប់ការប្រើប្រាស់សេវាបានដោយរបៀបណា,ដើម្បីបញ្ឈប់សេវានេះ អ្នកចាំបាច់ត្រូវចុះឈ្មោះឈប...


'តើលក្ខណសម្បត្តិអតិថិជនបែបណា ដើម្បីទទួលបានឥណទានសម្រាប់ទុនបង្វិល'

In [58]:
def formatting_prompts_func(examples):
    instructions = examples["input"]
    outputs      = examples["output"]
    texts = []
    for instruction, output in zip(instructions, outputs):
        # Strictly coerce elements to string and handle potential None values safely
        instruction_str = str(instruction) if instruction is not None else ""
        output_str = str(output) if output is not None else ""

        messages = [
            {
                "role": "user", "content": instruction_str.strip()
            },
            {
                "role": "assistant", "content": output_str.strip()
            }
        ]
        text = tokenizer.apply_chat_template(messages, add_generation_prompt=False, tokenize=False)
        texts.append(text)
    return {"text": texts}

### Map the Formatting Function to both Splits
Now we apply our formatting prompts to both our `'train'` and `'test'` splits.

In [59]:
# 1. Explicitly rename columns in both train and test splits to 'input' and 'output'
for split in dataset.keys():
    current_cols = dataset[split].column_names
    # Rename raw input column
    if "input,output" in current_cols:
        dataset[split] = dataset[split].rename_column("input,output", "input")
    # Rename raw index column to output
    if "__index_level_0__" in current_cols:
        dataset[split] = dataset[split].rename_column("__index_level_0__", "output")

# 2. Map formatting prompts across both splits
dataset = dataset.map(
    formatting_prompts_func,
    batched=True
)

print(dataset)

Map:   0%|          | 0/414 [00:00<?, ? examples/s]

Map:   0%|          | 0/104 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input', 'output', 'text'],
        num_rows: 414
    })
    test: Dataset({
        features: ['input', 'output', 'text'],
        num_rows: 104
    })
})


In [ ]:
dataset['train']['text'][0]

'<bos><start_of_turn>user\nធនាគារ អេស៊ីលីដា ភីអិលស៊ី ចេញលិខិតអះអាងលើគណនី និងព័ត៌មានអតិថិជន ទៅតាមការស្នើសុំជាលាយលក្ខណ៍អក្សររបស់អតិថិជន។ ធនាគារអាចចេញលិខិតប្រភេទនេះ បានតាមតម្រូវការរបស់អតិថិជន ឲ្យតែព័ត៌មានទាំងនោះជាព័ត៌មានពិតប្រាកដ និងជាក់ស្តែងនៃប្រតិបត្តិការផលិតផល-សេវាកម្ម ដែលអតិថិជនបាននិងកំពុងប្រើប្រាស់ជាមួយធនាគារ គិតត្រឹមកាលបរិច្ឆេទណាមួយជាក់លាក់នៅក្នុងអតីតកាល ឬបច្ចុប្បន្នកាល។<end_of_turn>\n<start_of_turn>model\nតើអតិថិជនអាចស្នើសុំឲ្យធនាគារចេញលិខិតអះអាងលើគណនី និងព័ត៌មានអតិថិជន លើព័ត៌មានអ្វីខ្លះ<end_of_turn>\n'

Let's see how row 100 looks like!

In [ ]:
dataset['train']['input'][0]

'ធនាគារ អេស៊ីលីដា ភីអិលស៊ី ចេញលិខិតអះអាងលើគណនី និងព័ត៌មានអតិថិជន ទៅតាមការស្នើសុំជាលាយលក្ខណ៍អក្សររបស់អតិថិជន។ ធនាគារអាចចេញលិខិតប្រភេទនេះ បានតាមតម្រូវការរបស់អតិថិជន ឲ្យតែព័ត៌មានទាំងនោះជាព័ត៌មានពិតប្រាកដ និងជាក់ស្តែងនៃប្រតិបត្តិការផលិតផល-សេវាកម្ម ដែលអតិថិជនបាននិងកំពុងប្រើប្រាស់ជាមួយធនាគារ គិតត្រឹមកាលបរិច្ឆេទណាមួយជាក់លាក់នៅក្នុងអតីតកាល ឬបច្ចុប្បន្នកាល។'

Let's see how the chat template did! Notice `Gemma-3` default adds a `<bos>`!

In [ ]:
dataset['train']['output'][00]

'តើអតិថិជនអាចស្នើសុំឲ្យធនាគារចេញលិខិតអះអាងលើគណនី និងព័ត៌មានអតិថិជន លើព័ត៌មានអ្វីខ្លះ'

<a name="Train"></a>
### Train the model
Now let's use Huggingface TRL's `SFTTrainer`! More docs here: [TRL SFT docs](https://huggingface.co/docs/trl/sft_trainer). We do 60 steps to speed things up, but you can set `num_train_epochs=1` for a full run, and turn off `max_steps=None`.

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments
from unsloth import is_bfloat16_supported

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset['train'],
    dataset_text_field = "text",
    #max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, # Can make training 5x faster for short sequences.
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 60,
        learning_rate = 2e-4,
        fp16 = not is_bfloat16_supported(),
        bf16 = is_bfloat16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
        report_to = "none", # Use this for WandB etc
    ),
)

Unsloth: Switching to float32 training since model cannot work with float16
Unsloth: We found double BOS tokens - we shall remove one automatically.


Unsloth: Tokenizing ["text"] (num_proc=2):   0%|          | 0/410 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


We also use Unsloth's `train_on_completions` method to only train on the assistant outputs and ignore the loss on the user's inputs. This helps increase accuracy of finetunes!

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<start_of_turn>user\n",
    response_part = "<start_of_turn>model\n",
)

Map:   0%|          | 0/410 [00:00<?, ? examples/s]

Let's verify masking the instruction part is done! Let's print the 100th row again:

In [ ]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<bos><start_of_turn>user\nទំហឥណទាន \\n ទំហំឥណទានអតិបរមា ១៥០០០០ដុល្លារអាមេរិក ឬសមមូលជាប្រាក់រៀល ។ \\n រយៈពេលខ្ចីអតិបរមា ៧២ខែ (ចិតសិបពីរខែ) ។<end_of_turn>\n<start_of_turn>model\nតើទំហំឥណទាន និងរយៈពេលខ្ចីអាចបានប៉ុន្មានដែរ<end_of_turn>\n'

Now let's print the masked out example - you should see only the answer is present:

In [ ]:
tokenizer.decode([tokenizer.pad_token_id if x == -100 else x for x in trainer.train_dataset[100]["labels"]]).replace(tokenizer.pad_token, " ")

'                                                                              តើទំហំឥណទាន និងរយៈពេលខ្ចីអាចបានប៉ុន្មានដែរ<end_of_turn> '

In [ ]:
# @title Show current memory stats
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = Tesla T4. Max memory = 14.563 GB.
4.381 GB of memory reserved.


Let's train the model! To resume a training run, set `trainer.train(resume_from_checkpoint = True)`

In [ ]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 410 | Num Epochs = 2 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 14,901,248 of 4,314,980,720 (0.35% trained)


Unsloth: Will smartly offload gradients to save VRAM!
Unsloth: Double buffering enabled (parallel H2D + compute) for backward pass.


Step,Training Loss
1,4.073515
2,3.955764
3,4.789878
4,3.813870
5,2.739128
6,2.406214
7,2.713283
8,2.511237
9,1.567807
10,0.990937


Unsloth: Restored added_tokens_decoder metadata in outputs/checkpoint-60/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in outputs/checkpoint-60.


In [ ]:
# @title Show final memory and time stats
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

307.648 seconds used for training.
5.13 minutes used for training.
Peak reserved memory = 4.887 GB.
Peak reserved memory for training = 0.506 GB.
Peak reserved memory % of max memory = 33.558 %.
Peak reserved memory for training % of max memory = 3.475 %.


<a name="Inference"></a>
### Inference
Let's run the model via Unsloth native inference! According to the `Gemma-3` team, the recommended settings for inference are `temperature = 1.0, top_p = 0.95, top_k = 64`

In [ ]:
from unsloth.chat_templates import get_chat_template
tokenizer = get_chat_template(
    tokenizer,
    chat_template = "gemma-3",
)
messages = [{
    "role": "user",
    "content": [{
        "type" : "text",
        "text" : "ទៅមុខ១០មហើយបត់ឆ្វេង",
    }]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)
outputs = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
)
tokenizer.batch_decode(outputs)

['<bos><start_of_turn>user\nទៅមុខ១០មហើយបត់ឆ្វេង<end_of_turn>\n<start_of_turn>model\nតើផ្លូវទៅមុខ៤០មហើយតើបត់ឆ្វេង<end_of_turn>']

 You can also use a `TextStreamer` for continuous inference - so you can see the generation token by token, instead of waiting the whole time!

In [ ]:
messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "ទៅមុខ១០ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

តើប្រើប្រាស់ប្រមានរ៉ាំងពាររថយន្ដតើធ្វើដូចម្តេច<end_of_turn>


<a name="Save"></a>
### Saving, loading finetuned models
To save the final model as LoRA adapters, either use Huggingface's `push_to_hub` for an online save or `save_pretrained` for a local save.

**[NOTE]** This ONLY saves the LoRA adapters, and not the full model. To save to 16bit or GGUF, scroll down!

In [ ]:
model.save_pretrained("gemma-3")  # Local saving
tokenizer.save_pretrained("gemma-3")
# model.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving
# tokenizer.push_to_hub("HF_ACCOUNT/gemma-3", token = "...") # Online saving

Unsloth: Restored added_tokens_decoder metadata in gemma-3/tokenizer_config.json.
Unsloth: Preserved sentencepiece asset `tokenizer.model` in gemma-3.


['gemma-3/processor_config.json']

Now if you want to load the LoRA adapters we just saved for inference, set `False` to `True`:

In [ ]:
if False:
    from unsloth import FastModel
    model, tokenizer = FastModel.from_pretrained(
        model_name = "lora_model", # YOUR MODEL YOU USED FOR TRAINING
        max_seq_length = 2048,
        load_in_4bit = True,
    )

messages = [{
    "role": "user",
    "content": [{"type" : "text", "text" : "បន្តទៅក្រោយ៩ម",}]
}]
text = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True, # Must add for generation
)

from transformers import TextStreamer
_ = model.generate(
    **tokenizer([text], return_tensors = "pt").to("cuda"),
    max_new_tokens = 64, # Increase for longer outputs!
    # Recommended Gemma-3 settings!
    temperature = 1.0, top_p = 0.95, top_k = 64,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

តើធ្វើដូចម្តេចបើកូនខ្ញុំបានគប្បីថតយូបាតអតិផឹករបស់បេសីនីអគិតទៅថាពិតជាប្រកបដោយចក្ខុវិស័យនិងរបៀបវារីអតិផឹក<end_of_turn>


### Saving to float16 for VLLM

We also support saving to `float16` directly for deployment! We save it in the folder `gemma-3-finetune`. Set `if False` to `if True` to let it run!

In [ ]:
if False: # Change to True to save finetune!
    model.save_pretrained_merged("gemma-3-finetune", tokenizer)

If you want to upload / push to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload finetune
    model.push_to_hub_merged(
        "HF_ACCOUNT/gemma-3-finetune", tokenizer,
        token = "hf_..."
    )

In [ ]:
if False: # Change to True to save to GGUF
    model.save_pretrained_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # For now only Q8_0, BF16, F16 supported
    )

Likewise, if you want to instead push to GGUF to your Hugging Face account, set `if False` to `if True` and add your Hugging Face token and upload location!

In [ ]:
if False: # Change to True to upload GGUF
    model.push_to_hub_gguf(
        "gemma-3-finetune",
        quantization_type = "Q8_0", # Only Q8_0, BF16, F16 supported
        repo_id = "HF_ACCOUNT/gemma-finetune-gguf",
        token = "hf_...",
    )